# 🥔 The Great Kacchi Aloo Mystery: Aloo Theory + EDA + CV

**Short answer:** guests go back for seconds when there are **1–2 potatoes per guest**.
- Below 1 per guest there isn't enough aloo to go round.
- Above 2 per guest there's too much potato and not enough meat and rice.

One ratio and two thresholds give about 95.5% CV accuracy. Tree models on the raw counts get about 86%.

This notebook:
1. Cleans the data traps from Mitu's field notes.
2. Shows why no raw column works and the per-guest ratio does.
3. Compares models with 10×5 repeated stratified CV.
4. Writes several candidate submissions (`submission_*.csv`). The final choice is made by CV.

In [ ]:
import glob, os, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.optimize import minimize
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
import lightgbm as lgb
warnings.filterwarnings('ignore')

DATA = os.path.dirname(sorted(glob.glob('/kaggle/input/**/train.csv', recursive=True) + glob.glob('../data/train.csv'))[0])
print('data dir:', DATA)
C1, C0, CT, INK = '#2a78d6', '#eb6834', '#8a8983', '#0b0b0b'
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True,
                     'axes.axisbelow': True, 'grid.color': '#e4e3df', 'figure.dpi': 110})

## 1. Cleaning the field-note traps

In [ ]:
BANGLA = str.maketrans('০১২৩৪৫৬৭৮৯', '0123456789')
NUM = ['guests','aloo_count','mutton_kg','borhani_glasses','fairy_lights','dhol_players','aunties_asking_when_marriage']
def load(name):
    d = pd.read_csv(f'{DATA}/{name}', dtype=str)
    d['bangla_digits'] = d.fairy_lights.str.contains('[০-৯]', regex=True)
    d['fairy_lights'] = d.fairy_lights.str.translate(BANGLA).astype(float)       # trap 2: Bangla digits
    for c in NUM: d[c] = pd.to_numeric(d[c])
    d['x10'] = d.aloo_count / d.guests > 3                                         # trap 3: extra zero
    d['aloo'] = np.where(d.x10, d.aloo_count / 10, d.aloo_count)
    d['apg'] = d.aloo / d.guests                                                   # aloo per guest
    return d
tr, te = load('train.csv'), load('test.csv')
tr['y'] = tr.went_back_for_seconds.astype(int)
print(pd.DataFrame({
    'missing aloo': [tr.aloo_count.isna().sum(), te.aloo_count.isna().sum()],
    'Bangla digits': [tr.bangla_digits.sum(), te.bangla_digits.sum()],
    'extra-zero aloo': [tr.x10.sum(), te.x10.sum()],
    'guests > 700': [(tr.guests > 700).sum(), (te.guests > 700).sum()]}, index=['train', 'test']))
tr = tr.dropna(subset=['apg']).reset_index(drop=True)    # trap 1: missing aloo can't be recovered (apg is independent of every other column)

- **Test is clean:** all the dirt (24 missing aloo values, 15 Bangla-digit values, 6 extra-zero potato counts) is in **train only**. Test is clean.
- **The extra-zero fix is safe:** clean aloo/guest never goes above 2.7, so anything above 3 has to be the ×10 typo.
- **Big test weddings:** 145 test weddings are larger than the biggest train wedding (trap 4).

## 2. No raw column works; the ratio does

In [ ]:
print('Spearman correlation with the target:')
print(tr[NUM].corrwith(tr.y, method='spearman').round(3).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
ax = axes[0]
bins = np.arange(0.4, 2.76, 0.05)
ax.hist([tr.apg[tr.y == 0], tr.apg[tr.y == 1]], bins=bins, stacked=True, color=[C0, C1],
        label=['no seconds', 'went back for seconds'], edgecolor='white')
for t in (1, 2): ax.axvline(t, color=INK, ls='--', lw=1.2)
ax.set_xlabel('aloo per guest'); ax.set_ylabel('weddings'); ax.legend()
ax.set_title('Seconds happen only between 1 and 2 aloo per guest', fontweight='bold')
ax = axes[1]
ax.scatter(te.guests, te.aloo, s=8, color=CT, alpha=.5, label='test')
ax.scatter(tr.guests, tr.aloo, s=8, c=np.where(tr.y == 1, C1, C0), label='train (colour = target)')
g = np.array([0, 1500]); ax.plot(g, g, 'k--', lw=1); ax.plot(g, 2 * g, 'k--', lw=1); ax.axvline(700, color=CT, ls=':')
ax.set_xlabel('guests'); ax.set_ylabel('aloo'); ax.legend(loc='upper left')
ax.set_title('In raw counts the band is a wedge that extends into the bigger test weddings', fontweight='bold')
plt.tight_layout(); plt.show()

**Why a ratio and not a tree model?**
- In raw counts the decision boundary is two diagonal lines, `aloo = guests` and `aloo = 2·guests`.
- Trees can only build axis-aligned staircases, and those stop at guests = 700, while 36% of test weddings are larger.
- Per-guest ratios have the same distribution in train and test, so a ratio-based rule needs no extrapolation.

## 3. Is the noise in the data or in the labels?

In [ ]:
al = pd.concat([tr, te])
for c, s in [('aloo/guest', al.apg), ('mutton/guest', al.mutton_kg / al.guests), ('borhani/guest', al.borhani_glasses / al.guests)]:
    print(f'{c:14s} min {s.min():.4f}  max {s.max():.4f}')
b = tr.groupby(pd.cut(tr.apg, np.arange(0.4, 2.81, 0.1)), observed=True).y.agg(['mean', 'count'])
print(b.round(2).T.to_string())

- **Uniform ratios with sharp bounds:** aloo/guest runs from 0.40 to 2.70, mutton/guest from 0.18 to 0.32, and borhani/guest from 0.80 to 1.50.
- **The counts aren't noisy:** if Tutul's potato counts carried noise, values would spill past those bounds, and they don't.
- **The leftover errors are label noise:** they sit in a narrow zone around each edge, where the label is roughly a coin flip.
- **Other columns don't help:** event, city, drone, mutton, borhani, dhol, aunties and wedding order were all tested and none predicts these flips. That puts the realistic ceiling at about 95–96%.

## 4. Models compared with 10×5 repeated stratified CV

In [ ]:
r, y, rt = tr.apg.values, tr.y.values, te.apg.values
band = lambda x, lo, hi: ((x > lo) & (x < hi)).astype(int)
sig = lambda z: 1 / (1 + np.exp(-np.clip(z, -50, 50)))

def fit_band(r, y):
    # accuracy-optimal (lo, hi); ties resolved to the centre of the best plateau
    lo_g, hi_g = np.arange(0.85, 1.15, 0.0025), np.arange(1.85, 2.2, 0.0025)
    H = r[None, :] < hi_g[:, None]; best, arg = -1, []
    for lo in lo_g:
        acc = (((r > lo)[None, :] & H).astype(int) == y).mean(1); m = acc.max()
        if m > best + 1e-12: best, arg = m, []
        if abs(m - best) < 1e-12: arg += [(lo, hi_g[j]) for j in np.where(acc == m)[0]]
    return np.array(arg).mean(0)

def fit_logmle(r, y):
    def nll(q):
        lo, hi, k = q; z = np.log(r)
        p = np.clip(sig(k * (z - np.log(lo))) * sig(k * (np.log(hi) - z)), 1e-9, 1 - 1e-9)
        return -(y * np.log(p) + (1 - y) * np.log(1 - p)).sum()
    return minimize(nll, [1, 2, 20], method='Nelder-Mead').x[:2]

def lgbm_pred(r_tr, y_tr, r_te, seeds=range(10)):
    P = np.mean([lgb.LGBMClassifier(n_estimators=300, learning_rate=0.02, num_leaves=4, min_child_samples=15,
                                    subsample=0.8, subsample_freq=1, random_state=s, verbose=-1)
                 .fit(np.log(r_tr)[:, None], y_tr).predict_proba(np.log(r_te)[:, None])[:, 1] for s in seeds], 0)
    return (P > .5).astype(int)

MODELS = {
    'band_rule':   lambda rt_, yt_, rv: band(rv, *fit_band(rt_, yt_)),
    'lgbm_logapg': lambda rt_, yt_, rv: lgbm_pred(rt_, yt_, rv, seeds=range(3)),
    'knn5':        lambda rt_, yt_, rv: KNeighborsClassifier(5).fit(np.log(rt_)[:, None], yt_).predict(np.log(rv)[:, None]),
    'logmle_band': lambda rt_, yt_, rv: band(rv, *fit_logmle(rt_, yt_)),
    'fixed_1_2':   lambda rt_, yt_, rv: band(rv, 1, 2),
}
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)
scores = {k: [] for k in MODELS}
for t, v in cv.split(r, y):
    for k, f in MODELS.items(): scores[k].append((f(r[t], y[t], r[v]) == y[v]).mean())
CV = pd.DataFrame({k: [np.mean(s), np.std(s) / np.sqrt(len(s))] for k, s in scores.items()}, index=['cv_acc', 'se']).T.sort_values('cv_acc', ascending=False)
print(CV.round(4))

## 5. Fit on all of train and write the candidate submissions

In [ ]:
lo, hi = fit_band(r, y); mlo, mhi = fit_logmle(r, y)
print(f'band rule thresholds: ({lo:.4f}, {hi:.4f})   log-MLE thresholds: ({mlo:.4f}, {mhi:.4f})')
preds = {
    'band_rule':   band(rt, lo, hi),
    'lgbm_logapg': lgbm_pred(r, y, rt),
    'knn5':        KNeighborsClassifier(5).fit(np.log(r)[:, None], y).predict(np.log(rt)[:, None]),
    'logmle_band': band(rt, mlo, mhi),
}
for k, p in preds.items():
    pd.DataFrame({'wedding_id': te.wedding_id, 'went_back_for_seconds': p}).to_csv(f'submission_{k}.csv', index=False)
    print(f'submission_{k}.csv  CV {CV.loc[k, "cv_acc"]:.4f}  positive rate {p.mean():.3f}  differs from band_rule on {(p != preds["band_rule"]).sum()} rows')
# Default submission.csv = best by CV
best = CV.index[0]
pd.DataFrame({'wedding_id': te.wedding_id, 'went_back_for_seconds': preds[best]}).to_csv('submission.csv', index=False)
print('submission.csv =', best)

## 🥔 The Aloo Theory, in plain words
- **Not enough aloo (under 1 per guest):** someone at the table gets no potato. Guests feel cheated and nobody goes back.
- **Just right (1 to 2 per guest):** everyone finds their golden potato and wants another plate.
- **Too much aloo (over 2 per guest):** the plate is mostly potato and too little mutton. Guests feel it's skimpy and don't go back.
- **Everything else is decoration:** fairy lights, dhol players, drones, aunties, city and event make no difference.
- **The thinking tip was right:** two thresholds on one ratio beat models fed every raw column.